# SS3 Part B — Fraud Detection for STADIOalot with the SS2 Hybrid Model
**CAP182 Capstone · applying the final method from SS2 to the real client data extract**

**Business problem (SS1):** STADIOalot loses money to fraudulent orders (card fraud, chargebacks). Fraud is rare, so the client needs a model that ranks every new order by fraud risk so analysts can review the riskiest ones first, with few false alarms.

**Final method from SS2 (Part C comparison and Part D recommendation): Model 2, the hybrid Isolation Forest + Neural Network.** It beat the Random Forest (Model 1) on PR-AUC (0.615 vs 0.335) on the SS2 benchmark data.
1. An **Isolation Forest** is fitted on *legitimate* training orders only, and produces three anomaly features: `anomaly_score`, `anomaly_flag`, `anomaly_bin`.
2. A **feed-forward neural network** (64 → 32 → 1, dropout 0.3, Adam, SMOTE 0.5 on training data only) is trained on the original features plus the three anomaly features and outputs the fraud probability.

Part D listed adaptations needed for STADIOalot's data (customer-based velocity features, chargeback + flag labels, contextual features, risk-tiering, leakage-safe preprocessing). They are applied here and flagged where used.

| # | Component | Marks |
|---|---|---|
| 1 | Dataset loading | 2 |
| 2 | Dataset preprocessing (incl. train / validation / test split) | 5 |
| 3 | Model training | 5 |
| 4 | Model validation (unseen test set) | 5 |
| 5 | Results visualisation | 3 |

> **POPIA:** name, email and phone are never loaded; date of birth is used only to derive an age. The data folder is git-ignored.

In [4]:
import os, json, zipfile, warnings, time
import numpy as np, pandas as pd
import matplotlib.pyplot as plt, seaborn as sns
import joblib
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'
import tensorflow as tf
from tensorflow import keras
from sklearn.ensemble import IsolationForest, RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from imblearn.over_sampling import SMOTE
from sklearn.metrics import (roc_auc_score, average_precision_score, precision_recall_curve, roc_curve,
                             confusion_matrix, precision_score, recall_score, f1_score)
warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid', context='notebook')
plt.rcParams.update({'figure.dpi': 110, 'axes.titleweight': 'bold'})
SEED = 42
np.random.seed(SEED); tf.keras.utils.set_random_seed(SEED)
os.makedirs('outputs', exist_ok=True)

ModuleNotFoundError: No module named 'imblearn'

---
# 1. DATASET LOADING
Only the tables relevant to fraud are loaded. The other tables (search history, abandoned carts, product listings, seller profiles, complaints) belong to other problems or cannot be linked to an order (`01_orders` has no `seller_id` / `listing_id`); `10_purchase_histories` is a lifetime summary that includes the orders being predicted, so using it would leak the answer.

| Table | Use |
|---|---|
| `01_orders.csv` | One row per order (the unit we score) |
| `02_payments.csv` | Amount, currency, payment method |
| `09_customer_demographics.csv` | Age, tenure, country, city and province only (PII columns are not read) |
| `03_chargebacks.csv`, `04_flagged_transactions.csv` | **Labels only** (post-event) |

**Data format justification:** CSV files read with pandas and joined on `order_id` / `customer_id` into one row per paid order. This matches the SS2 pipeline, which also worked on a flat one-row-per-transaction table.

In [ ]:
ZIP_PATH, DATA_DIR = 'stadioalot_data extract.zip', 'data/stadioalot'
if not os.path.exists(DATA_DIR):
    assert os.path.exists(ZIP_PATH), f'Put {ZIP_PATH} next to this notebook (or extract to {DATA_DIR})'
    with zipfile.ZipFile(ZIP_PATH) as z: z.extractall('data')

DEMO_COLS = ['customer_id', 'date_of_birth', 'location_city', 'location_state', 'location_country', 'account_creation_date']  # no PII
orders_raw = pd.read_csv(f'{DATA_DIR}/01_orders.csv')
pay_raw    = pd.read_csv(f'{DATA_DIR}/02_payments.csv')
demo_raw   = pd.read_csv(f'{DATA_DIR}/09_customer_demographics.csv', usecols=DEMO_COLS)
cb_raw     = pd.read_csv(f'{DATA_DIR}/03_chargebacks.csv')
flag_raw   = pd.read_csv(f'{DATA_DIR}/04_flagged_transactions.csv')
for n, d in [('orders', orders_raw), ('payments', pay_raw), ('demographics', demo_raw), ('chargebacks', cb_raw), ('flagged_transactions', flag_raw)]:
    print(f'{n:22s} rows={len(d):6d} cols={d.shape[1]}')

# Target. PRIMARY label 'is_fraud': fraudulent chargeback OR flag resolved as Blocked / Escalated to bank.
# Part D of SS2 also proposed 'flagged OR charged back' -> kept as ALTERNATIVE label 'is_fraud_broad' (sensitivity check in section 4).
fraud_ids = set(cb_raw.loc[cb_raw.chargeback_reason.str.strip().str.lower() == 'fraudulent', 'order_id']) | \
            set(flag_raw.loc[flag_raw.resolution.str.strip().isin(['Blocked', 'Escalated to bank']), 'order_id'])
broad_ids = set(cb_raw.order_id) | set(flag_raw.order_id)
print(f'Primary fraud orders: {len(fraud_ids)} | broad (any flag or chargeback): {len(broad_ids)}')
orders_raw.head()

### Dataset summary
* **Raw extract:** 8,963 order rows (17 exact duplicates) → **8,946 unique orders**, 2025-01-01 to 2026-09-06.
* **Modelling population:** **8,412 paid orders**. The 534 cancelled orders have no payment (no transaction took place) so they are not scored.
* **Classes:** binary, **0 = legitimate**, **1 = confirmed fraud** (fraudulent chargeback, or blocked / escalated-to-bank flag).
* **Counts:** 8,011 legitimate vs **401 fraud**.
* **Imbalance:** fraud is **4.77%** of orders, about **1 fraud per 20 legitimate orders**. This is far less extreme than the SS2 benchmark (0.39%), but still strongly imbalanced, so accuracy is meaningless and SS2's imbalance-aware metrics (PR-AUC, precision, recall) are used.

In [3]:
---
# 2. DATASET PREPROCESSING
Same four feature families as SS2 (temporal, contextual, behavioural velocity, customer history) plus risk-tiering of high-cardinality categoricals. Everything that *learns* from data (risk tiers, scaler, Isolation Forest, SMOTE, network) uses the **training subset only**.

SyntaxError: invalid syntax (4021806821.py, line 1)

In [ ]:
# ---- 2.1 Cleaning --------------------------------------------------------------------------------
orders = orders_raw.drop_duplicates().copy()
for d in (orders, pay_raw, demo_raw):
    for c in d.select_dtypes('object').columns:
        d[c] = d[c].astype(str).str.strip().replace({'nan': np.nan})
pay = pay_raw.copy()
pay['payment_method'] = pay['payment_method'].str.title().replace({'Cod': 'COD', 'Instant Eft': 'Instant EFT'})
pay['currency_code']  = pay['currency_code'].str.upper()
orders['order_dt'] = pd.to_datetime(orders['order_date'] + ' ' + orders['order_time'], errors='coerce')
orders['paid_dt']  = pd.to_datetime(orders['paid_date'], errors='coerce')
demo = demo_raw.copy()
demo['dob'] = pd.to_datetime(demo['date_of_birth'], errors='coerce'); demo['acct_created'] = pd.to_datetime(demo['account_creation_date'], errors='coerce')

# ---- 2.2 Currency -> ZAR (SS2 Part D: 'confirm all amounts are in ZAR'). Fixed approximate rates (assumption) ---
ZAR_RATE = {'ZAR': 1.0, 'USD': 18.0, 'EUR': 20.0, 'GBP': 23.0}
pay['amount_zar'] = pay['payment_amount'] * pay['currency_code'].map(ZAR_RATE)

# ---- 2.3 Join, labels, range checks -----------------------------------------------------------------
df = (orders.merge(pay[['order_id', 'payment_method', 'currency_code', 'amount_zar']], on='order_id', how='inner')
            .merge(demo[['customer_id', 'dob', 'acct_created', 'location_city', 'location_state', 'location_country']], on='customer_id', how='left'))
df['is_fraud'] = df.order_id.isin(fraud_ids).astype(int)
df['is_fraud_broad'] = df.order_id.isin(broad_ids).astype(int)
n0 = len(df); df = df[(df.amount_zar > 0) & (df.order_quantities > 0) & df.order_dt.notna()]
print(f'Dropped {n0-len(df)} rows failing range/format checks -> {len(df):,} paid orders')

In [ ]:
# ---- 2.4 Feature engineering (all point-in-time: only information known when the order was placed) ----
df = df.sort_values('order_dt').reset_index(drop=True)
# Temporal
df['trans_hour'] = df.order_dt.dt.hour; df['trans_dayofweek'] = df.order_dt.dt.dayofweek
df['is_weekend'] = (df.trans_dayofweek >= 5).astype(float); df['is_night'] = (df.trans_hour < 6).astype(float)
# Contextual
df['amt_log'] = np.log1p(df.amount_zar); df['unit_price_log'] = np.log1p(df.amount_zar / df.order_quantities)
df['age_years'] = ((df.order_dt - df.dob).dt.days / 365.25).clip(18, 100)
df['acct_tenure_days'] = (df.order_dt - df.acct_created).dt.days.clip(lower=0)
df['pay_delay_min'] = ((df.paid_dt - df.order_dt).dt.total_seconds() / 60).clip(lower=0)
df['is_foreign_ccy'] = (df.currency_code != 'ZAR').astype(float)
df['is_foreign_cust'] = (df.location_country.fillna('South Africa') != 'South Africa').astype(float)
df['has_comment'] = df.comments.notna().astype(float)          # SS2 Part D: use order comments (free text is only a delivery note, so a presence flag)
# Behavioural velocity per CUSTOMER (SS2 Part D: group by customer_id instead of card number). Backward-looking only.
def velocity(g):
    t = g.order_dt.values.astype('datetime64[s]').astype('int64'); a = g.amount_zar.values
    cnt, tot, gap = np.zeros(len(g)), np.zeros(len(g)), np.full(len(g), np.nan)
    for i in range(len(g)):
        m = (t[:i] >= t[i] - 86400)
        cnt[i], tot[i] = m.sum(), a[:i][m].sum()
        if i: gap[i] = (t[i] - t[i-1]) / 60
    return pd.DataFrame({'txn_count_24h': cnt, 'txn_amt_sum_24h': tot, 'mins_since_last_txn': gap}, index=g.index)
df = df.join(df.groupby('customer_id', group_keys=False).apply(velocity))
df['is_first_order'] = df.mins_since_last_txn.isna().astype(float)
# Customer history: z-score of this amount vs the customer's own PRIOR orders (shifted so the order itself is excluded)
g = df.groupby('customer_id')['amount_zar']
prior_n = g.cumcount(); prior_mean = (g.cumsum() - df.amount_zar) / prior_n.replace(0, np.nan)
prior_sq = (df.amount_zar**2).groupby(df.customer_id).cumsum() - df.amount_zar**2
prior_std = np.sqrt((prior_sq / prior_n.replace(0, np.nan) - prior_mean**2).clip(lower=0))
df['prior_orders'] = prior_n.astype(float)
df['cust_amt_zscore'] = ((df.amount_zar - prior_mean) / prior_std.replace(0, np.nan)).clip(-10, 10)
pm = pd.get_dummies(df.payment_method, prefix='pay').astype(float); df = pd.concat([df, pm], axis=1)
print('Engineered; example rows:'); df[['txn_count_24h', 'txn_amt_sum_24h', 'mins_since_last_txn', 'cust_amt_zscore', 'prior_orders']].describe().round(2).loc[['mean', 'max']]

In [ ]:
# ---- 2.5 Chronological 60 / 20 / 20 split (train = past, validation = next, test = most recent) ------------
n = len(df); i1, i2 = int(.6 * n), int(.8 * n)
train, val, test = df.iloc[:i1].copy(), df.iloc[i1:i2].copy(), df.iloc[i2:].copy()

# ---- 2.6 Risk-tier grouping of high-cardinality categoricals (SS2 / Carneiro 2016), fitted on TRAIN only ----
def fit_risk_map(col, label, min_n=15, tiers=4):
    st = train.groupby(col)[label].agg(['mean', 'count']); st = st[st['count'] >= min_n]
    return pd.qcut(st['mean'].rank(method='first'), tiers, labels=False).add(1).to_dict()
def add_risk(part, maps):
    for col, mp in maps.items(): part[col + '_risk'] = part[col].map(mp).fillna(2.5)       # unseen / rare -> neutral tier
    return part
NUM = ['trans_hour', 'trans_dayofweek', 'is_weekend', 'is_night', 'amt_log', 'unit_price_log', 'order_quantities', 'age_years',
       'acct_tenure_days', 'pay_delay_min', 'is_foreign_ccy', 'is_foreign_cust', 'has_comment', 'txn_count_24h', 'txn_amt_sum_24h',
       'mins_since_last_txn', 'is_first_order', 'prior_orders', 'cust_amt_zscore']
BASE_FEATURES = NUM + ['location_city_risk', 'location_state_risk'] + list(pm.columns)

def make_matrices(label):
    # Everything learned (risk tiers, medians, scaler) is fitted on train only and re-used on validation and test.
    maps = {c: fit_risk_map(c, label) for c in ['location_city', 'location_state']}
    parts = [add_risk(p.copy(), maps) for p in (train, val, test)]
    med = parts[0][BASE_FEATURES].median()
    sc = StandardScaler().fit(parts[0][BASE_FEATURES].fillna(med))
    X = [sc.transform(p[BASE_FEATURES].fillna(med)) for p in parts]
    y = [p[label].values for p in parts]
    return X, y, dict(maps=maps, medians=med, scaler=sc)

(X_train, X_val, X_test), (y_train, y_val, y_test), prep_artifacts = make_matrices('is_fraud')
split_tbl = pd.DataFrame({'orders': [len(train), len(val), len(test)], 'fraud': [y_train.sum(), y_val.sum(), y_test.sum()],
                          'fraud_%': [100*y_train.mean(), 100*y_val.mean(), 100*y_test.mean()],
                          'from': [p.order_dt.min().date() for p in (train, val, test)], 'to': [p.order_dt.max().date() for p in (train, val, test)]},
                         index=['train', 'validation', 'test']).round(2)
print(len(BASE_FEATURES), 'base features'); split_tbl

In [ ]:
### Why this preprocessing?
| Step | Reason |
|---|---|
| **De-duplicate, strip / standardise text** | The client README warns the extract is uncleaned; duplicates distort customer history and velocity. |
| **Drop cancelled orders** | No payment means no transaction to score. |
| **Convert to ZAR** | Amounts are in each row's own currency. SS2 Part D required resolving this *before* modelling, otherwise an unconverted 120 GBP looks tiny next to 2,000 ZAR. |
| **log-transform amounts** | Right-skewed order values (same as SS2's `amt_log`). |
| **Temporal features** (hour, weekday, weekend, night) | SS2 temporal family; fraud often clusters at odd hours. |
| **Customer velocity** (orders / spend in trailing 24h, minutes since last order) | SS2's second most important family in Al-Hajabed (2026). Part D said to regroup by `customer_id` instead of card number. Backward-looking only, so no future leakage. |
| **Customer z-score vs own history** | Fraud is often a break from the customer's own pattern; uses earlier orders only. The lifetime `10_purchase_histories` table is *not* used because it contains the order being predicted. |
| **Risk tiers for city and province** | SS2 used train-only fraud-rate tiers (Carneiro, 2016) instead of hundreds of one-hots. Unseen or rare places get a neutral tier. |
| **`has_comment` flag** | SS2 Part D proposed using Comments. These are delivery notes (e.g. "Ring the bell twice"), not reviews, so sentiment is meaningless and only presence is kept. |
| **`star_ratings` excluded** | Part D proposed Star Ratings, but the client README shows they are given *after* delivery, so they would leak the outcome and would not exist at scoring time. |
| **Excluded outcome tables / fields** (flags, chargebacks, resolutions, delivery, shipped date) | All post-event; using them as features would let the model see the answer. |
| **Drop name / email / phone, derive age from DOB** | POPIA data minimisation. |
| **Chronological 60/20/20 split** | SS2 used the benchmark's own train/test files; here the model must predict the *future*, so time order is used. A validation set is added for early stopping and threshold selection so the test set is touched once. |
| **Imputation, risk tiers, scaler fitted on train only** | Follows SS2's leakage-prevention rule. |
| **SMOTE only inside training (section 3)** | As in SS2, validation and test keep their natural class balance so metrics are realistic. |